# Sanskrit Studio - Colab worker
**Not yet run on real Colab by the author; expect to adjust pip/torch steps.** See COLAB.md.

Run cells 1-5 in order (setup). Then run **either** Mode A (Drive polling worker) **or** Mode B (manual bundle), not both.

## 1. Check GPU

In [ ]:
import subprocess, sys
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise SystemExit("NO GPU. Runtime > Change runtime type > T4 GPU, then rerun.")

## 2. Mount Drive and set the drive root

In [ ]:
#@title Drive root { run: "auto" }
DRIVE_FOLDER = "sanskrit_studio" #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
R = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
print("R =", R)

## 3. Verify required files on Drive

In [ ]:
from pathlib import Path
need = ["models/voice_steer_ema_2026-06-17.pt", "models/voice_armA_ema_2026-06-11.pt",
        "models/voc_bigvgan_EMA_2026-06-11.pth", "models/vocab.txt",
        "code/backend/__init__.py", "code/vagdhenu/src", "code/BigVGAN/configs"]
missing = [p for p in need if not (Path(R)/p).exists()]
for p in need: print("OK     " if p not in missing else "MISSING", p)
Path(R, "jobs").mkdir(exist_ok=True)
if missing: raise SystemExit("Missing files above. See COLAB.md step 1-3.")

## 4. Install dependencies and copy code (do NOT reinstall torch)

In [ ]:
# torch/torchaudio come preinstalled with Colab. The repo pins torch 2.8; Colab's version may differ.
!pip install -q "f5-tts @ git+https://github.com/ai4bharat/IndicF5.git@13f7c4d627cc10111aea8fe9c0039462cacacdc7" \
    vocos "x-transformers>=2.19.7" indic-transliteration librosa "soundfile>=0.12.1" \
    safetensors huggingface_hub "pyarrow<20" scipy python-dotenv
import torch, torchaudio
print("torch", torch.__version__, "torchaudio", torchaudio.__version__, "cuda", torch.cuda.is_available())

In [ ]:
import os, sys, shutil
APP = "/content/app"
shutil.rmtree(APP, ignore_errors=True)
shutil.copytree(f"{R}/code", APP)                        # fast local copy
os.makedirs(f"{APP}/vagdhenu", exist_ok=True)
if not os.path.exists(f"{APP}/vagdhenu/models"):
    os.symlink(f"{R}/models", f"{APP}/vagdhenu/models")  # weights stay on Drive
os.environ["SS_VAGDHENU"] = f"{APP}/vagdhenu"
os.environ["SS_BIGVGAN"] = f"{APP}/BigVGAN"
os.environ["SS_STORAGE"] = "/content/ss_storage"
os.chdir(APP)
if APP not in sys.path: sys.path.insert(0, APP)
MODELS = f"{R}/models"
print("app copied to", APP)

## 5. Smoke check: Engine imports

In [ ]:
import torch
assert torch.cuda.is_available(), "torch sees no CUDA"
print(torch.cuda.get_device_name(0))
from backend.engine import synth
from backend.app import config as C
print("engine OK; MODELS =", C.MODELS, "vocab exists:", (C.MODELS/"vocab.txt").exists())

## MODE A - Drive polling worker
Leave running; start jobs from the app. Stop with the interrupt (stop) button.

In [ ]:
# MODE A
!cd /content/app && python -u -m backend.worker.colab_worker --drive-root "$R" --models-dir "$MODELS"

## MODE B - manual bundle
Use instead of Mode A. Download the bundle zip from the app's Generate page, upload it here.

In [ ]:
# MODE B step 1: upload bundle.zip
from google.colab import files
import os, shutil, zipfile
up = files.upload()
name = next(iter(up))
B = "/content/bundle"
shutil.rmtree(B, ignore_errors=True)
with zipfile.ZipFile(name) as z: z.extractall(B)
# if the zip holds a single top-level folder, use it
items = os.listdir(B)
if len(items) == 1 and os.path.isdir(f"{B}/{items[0]}"): B = f"{B}/{items[0]}"
print("bundle dir:", B, os.listdir(B))

In [ ]:
# MODE B step 2: run it
!cd /content/app && python -u -m backend.worker.colab_worker --bundle "$B" --models-dir "$MODELS"

In [ ]:
# MODE B step 3: zip result/ and download
import shutil
from google.colab import files
out = shutil.make_archive("/content/result", "zip", B, "result")
files.download(out)